[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/internshipinabook/cv-internship-in-a-book/blob/main/notebooks/week3_preprocessing_STARTER.ipynb)


# Week 3 -- Preprocessing — Preparing Images for a Model (STARTER)
### The Computer Vision Internship | MediVision AI Lagos

**This week:** normalisation, ImageNet statistics, domain-specific stats, augmentation pipeline

**Dataset:** All five datasets (preprocessing)

**STARTER notebook** -- your working environment. Complete each exercise before moving on.


In [ ]:
# -- Colab/Local Setup -- run this first in Colab, skip locally -------------
import os

if not os.path.exists('requirements.txt'):
    # Clone the full course repository
    !git clone https://github.com/internshipinabook/cv-internship-in-a-book.git cv-book
    %cd cv-book
    # Install all required packages (suppress verbose output with -q)
    !pip install -r requirements.txt -q

# Create working directories
os.makedirs('outputs',  exist_ok=True)   # charts, reports, predictions
os.makedirs('models',   exist_ok=True)   # trained model checkpoints
os.makedirs('datasets', exist_ok=True)   # raw downloaded data
print('Setup complete.')


In [ ]:
# -- Reproducibility Seeds --------------------------------------------------
# Setting seeds ensures every random operation produces the same result
# on every run. Essential for: comparing runs, debugging, sharing results.

import random    # Python built-in random
import numpy as np   # NumPy random (data loading, sklearn)
import torch     # PyTorch random (neural networks, dropout)

SEED = 42        # 42 is conventional -- any integer works

random.seed(SEED)           # fix Python random() calls
np.random.seed(SEED)        # fix np.random.* calls
torch.manual_seed(SEED)     # fix PyTorch CPU operations

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)  # fix GPU operations too

# Prefer deterministic cuDNN algorithms (may slow training ~5%)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Seeds fixed: {SEED} | Device: {device}')


## Learning Objectives

By the end of this week, you will be able to:

- Build a reusable preprocessing pipeline with documented decisions for every transformation
- Implement domain-specific normalisation for all five datasets
- Write a custom PyTorch Dataset class for at least two domains
- Apply stain normalisation (Macenko method) to a BreakHis histopathology sample
- Save the complete preprocessing pipeline as a reusable module



---

## MONDAY -- "Resize, Crop, and Aspect Ratio"


All five datasets have different native sizes. Your model expects a fixed input size. The choice — resize to 224×224, or crop to 224×224 — is not arbitrary. For Oxford Pets, centred crop works because the subject is usually centred. For chest X-rays, the heart is not centred — centred crop may exclude the left lung base. For microscopy patches, the patch size is fixed at 460×700; resizing without preserving aspect ratio distorts cell morphology.


### Exercise 3.1 -- Transform pipeline

Build domain-appropriate transforms for all five datasets. For each, document: resize strategy, crop strategy, normalisation parameters, augmentations applied, and one risk of each augmentation choice.


In [ ]:
# Exercise 3.1: Transform pipeline
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Monday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Monday: Resize, Crop, and Aspect Ratio (scaffold) --
import torchvision.transforms as T

# Domain-appropriate resize strategies
PETS_TRANSFORM = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

XRAY_TRANSFORM = T.Compose([
    T.Resize((224,224)),   # Direct resize — preserves lung fields
    T.Grayscale(num_output_channels=3),  # 3-channel for pretrained models
    T.ToTensor(),
    T.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5]),  # X-ray specific
])


### Monday Morning Moment

*Slack — Monday, 3:00pm.*

**Ngozi Eze-Okafor:** Question — for the BreakHis dataset, should I use CenterCrop or RandomCrop?

**Yewande Adeyemi:** RandomCrop during training, CenterCrop at inference. Standard practice.

**Dr. Kwame Asante:** Wait. What is the patch size in BreakHis?

**Ngozi Eze-Okafor:** 460×700 at 40× magnification.

**Dr. Kwame Asante:** And what does the edge of a BreakHis patch contain?

**Ngozi Eze-Okafor:** Uh... I'll check. Background tissue? Slide boundary?

**Dr. Kwame Asante:** Correct. The edges of histopathology patches often contain slide preparation artefacts, not tumour tissue. RandomCrop on a 460×700 patch that crops to 224×224 will sometimes include artefact and exclude tumour. Write that in your preprocessing log.

**Yewande Adeyemi:** I had not thought about that.

**Dr. Kwame Asante:** That is why we have preprocessing logs.




---

## TUESDAY -- "Custom Dataset Classes"


torchvision.datasets.ImageFolder works when all images of one class are in one folder. BreakHis is not organised this way — each slide has a folder, within which are magnification subfolders. TCGA patches are organised by case ID. You need custom Dataset classes.


### Exercise 3.2 -- Custom Dataset class

Write BreakHisDataset from scratch. It must: handle the nested folder structure, accept a magnification argument (40X/100X/200X/400X), support train/val/test splits by slide ID (not by image), and apply transforms correctly.


In [ ]:
# Exercise 3.2: Custom Dataset class
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Tuesday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Tuesday: Custom Dataset Classes (scaffold) --
from torch.utils.data import Dataset
from PIL import Image
import os

class BreakHisDataset(Dataset):
    """BreakHis breast cancer histopathology dataset.
    
    Handles the nested folder structure:
    breakhis/
      malignant/
        slide_001/
          40X/  100X/  200X/  400X/
      benign/
        ...
    """
    def __init__(self, root, magnification="40X", transform=None, split="train"):
        self.samples = []
        self.labels  = []
        self.transform = transform
        for label_idx, label_name in enumerate(["benign", "malignant"]):
            pattern = os.path.join(root, label_name, "**", magnification, "*.png")
            paths = glob.glob(pattern, recursive=True)
            self.samples.extend(paths)
            self.labels.extend([label_idx] * len(paths))
    
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        img = Image.open(self.samples[idx]).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, self.labels[idx]



---

## WEDNESDAY -- "Stain Normalisation — The Abuja Problem"


Histopathology slides from different hospitals look different because different labs use different concentrations of haematoxylin and eosin stain. The Macenko method decomposes a slide into its stain components and normalises each to a reference slide. Without stain normalisation, a model trained on slides from Lagos General Hospital fails on slides from LUTH — not because the tumours differ, but because the purple is a different shade.


### Exercise 3.3 -- Stain normalisation audit

Apply Macenko normalisation to 10 slides from two different magnification levels. Do the normalised slides look more similar than the raw slides? Compute per-channel mean before and after. Report: did normalisation reduce the between-magnification variance?


In [ ]:
# Exercise 3.3: Stain normalisation audit
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Wednesday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Wednesday: Stain Normalisation — The Abuja Problem (scaffold) --
from torchstain import MacenkoNormalizer
from PIL import Image
import torchvision.transforms.functional as TF

# Load reference slide (choose a well-stained, representative image)
ref_slide = Image.open("datasets/breakhis/reference_slide.png").convert("RGB")
normalizer = MacenkoNormalizer()
normalizer.fit(TF.to_tensor(ref_slide).unsqueeze(0))

# Normalise a new slide
target = Image.open("datasets/breakhis/malignant/slide_002/40X/sample.png").convert("RGB")
norm_tensor = normalizer.normalize(TF.to_tensor(target).unsqueeze(0))
print("Stain normalisation applied ✅")



---

## THURSDAY -- "Video Preprocessing — Frame Sampling"


A video is a sequence of frames. For a 10-second clip at 25fps, that is 250 frames. Training on all 250 frames per clip is computationally prohibitive. Frame sampling selects a fixed number of frames per clip — uniformly, randomly, or at key positions. The choice affects what temporal information the model sees.


### Exercise 3.4 -- Frame sampling comparison

Sample 8 frames from one UCF-101 clip using uniform, random, and first-N strategies. Display all three as image grids. Which strategy captures the most informative moments for action recognition?


In [ ]:
# Exercise 3.4: Frame sampling comparison
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Thursday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Thursday: Video Preprocessing — Frame Sampling (scaffold) --
import cv2
import torch

def sample_frames(video_path, n_frames=16, mode="uniform"):
    """Extract n_frames from a video clip.
    mode: 'uniform' samples evenly; 'random' samples randomly.
    """
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if mode == "uniform":
        indices = torch.linspace(0, total-1, n_frames).long().tolist()
    else:
        indices = sorted(torch.randint(0, total, (n_frames,)).tolist())
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret: frames.append(frame)
    cap.release()
    return frames  # list of HWC uint8 numpy arrays



---

## FRIDAY -- "The Friday Build: preprocessing.py"


Save a single preprocessing module that any week can import. It must contain: domain transforms for all five datasets, the BreakHis and TCGA Dataset classes, the Macenko normaliser wrapper, and the video frame sampler. The preprocessing log documents every decision, its justification, and its risk.


**Friday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Friday: The Friday Build: preprocessing.py (scaffold) --
from preprocessing import (
    get_transform, BreakHisDataset, TCGADataset,
    StainNormalizer, sample_frames
)
# Every subsequent week starts with this import


### Friday Workplace Moment

*MediVision AI — Friday standup.*

**Dr. Kwame Asante:** Preprocessing log. Walk me through the most important decision.

**Ngozi Eze-Okafor:** Stain normalisation on BreakHis. Decision: apply Macenko normalisation using a reference slide from the Lagos General Hospital dataset. Justification: reduces inter-hospital stain variability. Risk: if the reference slide is not representative, normalisation degrades all slides toward an unrepresentative baseline.

**Dr. Kwame Asante:** How would you test whether the reference slide is representative?

**Ngozi Eze-Okafor:** Compute the Fréchet distance between the normalised distribution and the raw distribution of each hospital subset. If one hospital's slides move further from the reference than others, the reference is biased toward a specific lab's staining protocol.

**Dr. Kwame Asante:** Write that as Week 8's Monday exercise.



## YOUR WEEK 3 CHECKLIST

Check each item before moving to the next week.
If you cannot explain an item without notes, revisit that section.

- [ ] I have built and documented domain-appropriate transforms for all five datasets.
- [ ] My BreakHisDataset class splits by slide ID — not by image — to prevent data leakage.
- [ ] I can explain what stain normalisation does and name one risk of choosing a bad reference slide.
- [ ] I know why RandomCrop on histopathology patches may include slide artefacts.
- [ ] My preprocessing.py is importable and tested from a fresh notebook.


## Challenge Task

> **Core path:** Implement the Macenko normalisation algorithm from scratch (without torchstain). The key steps: SVD on the OD (optical density) image, extract stain vectors, normalise to a reference. Compare your output to torchstain's output on 5 slides.

> **Advanced path:** Test whether splitting BreakHis by image (random 80/20) vs by slide ID produces different test set F1 scores on a simple baseline. Quantify the data leakage effect.


## Common Mistakes

**Splitting histopathology data by image instead of by slide:** Multiple patches from the same slide will appear in both training and test sets. The model memorises slide-level features and overfits. Always split by slide ID.

**Applying augmentations at inference time:** Augmentation is a training-time technique. At inference, use deterministic transforms only. RandomHorizontalFlip at inference produces different predictions on the same image each time.

**Using a biased reference slide for Macenko normalisation:** The reference slide determines the target colour space. If it is from one specific lab's staining protocol, all slides are normalised toward that lab's artefacts.



---

## Get the Full Book

This notebook is part of **The Computer Vision Internship** -- Book 3 of 9, InternshipInABook(tm).

Covers: natural images, medical X-rays, breast & uterine cancer, video, GradCAM/LIME/SHAP/Integrated Gradients, and fairness audits.

Available at [internshipinabook.com](https://internshipinabook.com)
